# RESUME — 續跑未完成的訓練

當上一場 session 沒跑滿 `EPOCHS` 就結束時用這一份接下去。**只需要編輯下一個 cell。**

### 什麼情況會需要它
- `train_ablation.ipynb` 的 `STOP_AFTER_EPOCHS` 或 `DEADLINE_HOURS` 安全閥觸發
- Kaggle session 被中斷

### 執行前確認
1. 上一場的 `runs_{ARM}.zip` 已上傳成 Kaggle Dataset，路徑填進 `RESUME_SRC`。
2. `ARM` 與 `EPOCHS` **必須與上一場完全相同** —— `resume=True` 會沿用 checkpoint 內
   記錄的總輪數與全部超參數，這兩個值只是用來定位 run 目錄與還原注入狀態。
3. Internet 開啟（要抓 `v9_modules.py`）。

> `resume=True` 之後**不要再傳任何訓練參數**。也**不要**改用 `time=` 限制時數 ——
> Ultralytics 會用實測 epoch 時間反推並改寫 `args.epochs`（`engine/trainer.py:619`），
> 續跑的總輪數會被改掉。

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 唯一需要編輯的 cell
# ══════════════════════════════════════════════════════════════════════
ARM    = "A0"       # 必須與上一場相同
EPOCHS = 160        # 必須與上一場相同

DATASET_SRC = "/kaggle/input/datasets-yolo26-v5r"
RESUME_SRC  = "/kaggle/input/v5r-a0-160e/runs_A0.zip"   # 上一場的 runs zip 或解開後的資料夾

# 指定要從哪個 checkpoint 續跑。None = 自動挑 epoch 最大的可用檔案。
# ⚠ 若 RESUME_SRC 裡混有「失敗那一場」留下的 epochN.pt，自動挑選會挑到壞的那個，
#   這時必須在這裡寫死檔名，例如 "epoch100.pt"。
CKPT_NAME = None

# Step.1 環境

In [ ]:
!nvidia-smi
# 版本必須釘死：整套注入機制建立在 8.4.121 的 parse_model 與 BboxLoss 實作細節上
!pip install -q ultralytics==8.4.121

import ultralytics
assert ultralytics.__version__ == "8.4.121", \
    f"ultralytics 版本不符：{ultralytics.__version__}，注入機制可能失效"
print(f"▷ ultralytics {ultralytics.__version__}")

In [ ]:
# 取得 v9_modules.py —— 自訂 loss、自訂模組、六臂定義的單一真實來源。
# 直接抓 GitHub 上的最新版，避免 notebook 內再出現一份複製貼上（問題 C3）。
import subprocess, sys

URL = ("https://raw.githubusercontent.com/OneLeaf-dx/"
       "detection/main/Train%20Code/v9/v9_modules.py")
subprocess.run(["curl", "-sSLf", "-o", "/kaggle/working/v9_modules.py", URL], check=True)

sys.path.insert(0, "/kaggle/working")
import v9_modules as v9

assert v9.REQUIRED_ULTRALYTICS == ultralytics.__version__, \
    f"v9_modules 要求 ultralytics {v9.REQUIRED_ULTRALYTICS}"
assert ARM in v9.ARMS, f"未知的臂 {ARM}，可用：{list(v9.ARMS)}"
print(f"▷ v9_modules {v9.__version__}\n")
print(v9.arm_summary(ARM, EPOCHS))

# Step.2 資料集準備

In [ ]:
import os, shutil, sys, yaml

def render_progress_bar(current, total, task_name="檔案同步複製中", bar_length=25):
    percent = (current / total) * 100 if total > 0 else 100.0
    filled = int(bar_length * current // total) if total > 0 else bar_length
    bar = "█" * filled + "░" * (bar_length - filled)
    sys.stdout.write(f"\r▷ 正在執行 [{task_name}] | 進度: [{bar}] {percent:5.1f}% ({current}/{total})")
    sys.stdout.flush()


def copy_and_verify_dataset(src_dir, dst_dir):
    if not os.path.exists(src_dir):
        print(f"▷ 錯誤：找不到來源資料集目錄 {src_dir}")
        return False

    src_files = []
    for root, _, files in os.walk(src_dir):
        for file in files:
            src_files.append(os.path.relpath(os.path.join(root, file), src_dir))
    total_files = len(src_files)
    print(f"▷ 來源資料集掃描完成，共計 {total_files} 個檔案")

    for idx, rel_path in enumerate(src_files, 1):
        dst_path = os.path.join(dst_dir, rel_path)
        os.makedirs(os.path.dirname(dst_path), exist_ok=True)
        shutil.copy2(os.path.join(src_dir, rel_path), dst_path)
        if idx % 200 == 0 or idx == total_files:
            render_progress_bar(idx, total_files)

    print("\n\n▷ 正在檢查複製檔案")
    dst_files_set = set()
    for root, _, files in os.walk(dst_dir):
        for file in files:
            dst_files_set.add(os.path.relpath(os.path.join(root, file), dst_dir))

    missing, corrupted = [], []
    for rel_path in src_files:
        if rel_path not in dst_files_set:
            missing.append(rel_path)
        elif os.path.getsize(os.path.join(src_dir, rel_path)) != \
                os.path.getsize(os.path.join(dst_dir, rel_path)):
            corrupted.append(rel_path)

    print("≡" * 60)
    print("▷ 資料集複製完整性校驗：")
    print(f"  ▶ 來源檔案總數 : {len(src_files)}")
    print(f"  ▶ 目標檔案總數 : {len(dst_files_set)}")
    print(f"  ▶ 遺漏檔案數   : {len(missing)}")
    print(f"  ▶ 損毀/大小不符: {len(corrupted)}")
    ok = not missing and not corrupted
    print("▷ 檢查通過" if ok else f"▷ 檢查失敗  遺漏={missing[:5]}  損毀={corrupted[:5]}")
    print("≡" * 60)
    return ok


DST = "/kaggle/working/datasets-yolo26-v5r"
DATA_YAML = "/kaggle/working/data.yaml"

assert copy_and_verify_dataset(DATASET_SRC, DST), "資料集複製失敗，不要往下跑"

# data.yaml 改寫成絕對路徑（來源版本用的是相對的 path: .）
with open(os.path.join(DST, "data.yaml"), encoding="utf-8") as f:
    ycfg = yaml.safe_load(f)
ycfg.update(path=DST, train="train/images", val="valid/images", test="test/images")
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(ycfg, f, default_flow_style=False, allow_unicode=True)
assert ycfg["nc"] == 8, f"nc={ycfg['nc']}，應為 8"

# BOM 與舊快取會讓 Ultralytics 的標註解析出錯
bom_fixed = cache_removed = 0
for subdir, _, files in os.walk(DST):
    for file in files:
        path = os.path.join(subdir, file)
        if file.endswith(".cache"):
            os.remove(path)
            cache_removed += 1
        elif file.endswith(".txt") and "labels" in subdir:
            with open(path, "rb") as fh:
                is_bom = fh.read(3) == b"\xef\xbb\xbf"
            if is_bom:
                with open(path, encoding="utf-8-sig") as fh:
                    content = fh.read()
                with open(path, "w", encoding="utf-8") as fh:
                    fh.write(content)
                bom_fixed += 1

print(f"▷ data.yaml → {DATA_YAML}   nc={ycfg['nc']}")
print(f"▷ names = {ycfg['names']}")
print(f"▷ 修正 BOM {bom_fixed} 個、清除快取 {cache_removed} 個")
print("▷ Step.2 完成")

# Step.3 還原該臂的注入狀態
### 續跑必須讓 loss 與模組與第一場完全一致，否則模型會靜默改變。

In [ ]:
import yaml as _yaml
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils.torch_utils import get_flops, get_num_params

# 依 ARMS 表裝好該臂需要的 loss / 模組，並取得超參數
HP = v9.install_arm(ARM, epochs=EPOCHS)

cfg = v9.make_arm_yaml(nc=ycfg["nc"], **v9.ARMS[ARM]["arch"])
# 檔名必須帶 scale 字母：yaml_model_load 會用檔名覆寫 dict 裡的 scale 鍵，
# 少了 "n" 會落回 scales 字典的第一項並印出 warning —— 目前碰巧正確，但很脆弱
YAML_PATH = f"/kaggle/working/yolo26n-p2-{ARM}.yaml"
with open(YAML_PATH, "w", encoding="utf-8") as f:
    _yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

_m = DetectionModel(cfg=cfg, ch=3, nc=ycfg["nc"], verbose=False)
print(f"▷ {ARM}: {get_num_params(_m):,} params / {get_flops(_m, imgsz=640):.2f} GFLOPs @640")
print(f"▷ yaml → {YAML_PATH}")
print("▷ 超參數：" + "  ".join(f"{k}={v}" for k, v in sorted(HP.items())))
del _m

# Step.R 續跑

In [ ]:
import os, re, shutil, zipfile
import torch
import ultralytics.nn.tasks
import ultralytics.utils.loss
from ultralytics import YOLO

# ── 0. 前置檢查：注入狀態必須與該臂的定義相符 ────────────────────────
# 舊版這裡硬寫「必須有自訂 loss 與 StarTripletBlock」，但那只對 v9 原始設定成立；
# A0 兩者都沒有，會直接 assert 失敗。改成依 ARMS 表逐項核對。
spec = v9.ARMS[ARM]
patched = ultralytics.utils.loss.bbox_iou.__name__ == "bbox_wise_inner_mpdiou"
assert patched == (spec["loss_ratio"] is not None), (
    f"{ARM} 的自訂 loss 安裝狀態不符（patched={patched}）—— 請先執行 Step.3")
if spec["arch"]["stb"]:
    assert ultralytics.nn.tasks.C2f is v9.StarTripletBlock, "C2f 別名未生效 —— 請先執行 Step.3"
print(f"▷ 0/4 {ARM} 的注入狀態正確："
      f"loss={'自訂 ratio=' + str(spec['loss_ratio']) if patched else '內建 CIoU'}，"
      f"STB={'有' if spec['arch']['stb'] else '無'}")

# ── 1. 還原上一場的整個 runs 目錄 ─────────────────────────────────────
RUN_NAME = f"v5r_{ARM}_{EPOCHS}e"          # 與 install_arm 產生的 name 一致
WORK_RUNS = "/kaggle/working/runs"
WDIR = f"{WORK_RUNS}/detect/{RUN_NAME}/weights"

assert os.path.exists(RESUME_SRC), f"找不到 {RESUME_SRC}，請確認 Kaggle Dataset 路徑"
os.makedirs(WORK_RUNS, exist_ok=True)
if RESUME_SRC.endswith(".zip"):
    with zipfile.ZipFile(RESUME_SRC) as z:
        z.extractall(WORK_RUNS)
else:
    shutil.copytree(RESUME_SRC, WORK_RUNS, dirs_exist_ok=True)
assert os.path.isdir(WDIR), (
    f"還原後找不到 {WDIR}。runs zip 的內層應為 detect/{RUN_NAME}/weights/...，"
    "若結構不同請調整 RESUME_SRC 或解壓目標。")
print(f"▷ 1/4 已還原 runs 目錄：{sorted(os.listdir(WDIR))}")

# ── 2. 挑出可續跑的 checkpoint ────────────────────────────────────────
# final_eval() 會對 last.pt / best.pt 執行 strip_optimizer()，把 epoch 改成 -1
# 並清掉 optimizer/EMA，那種檔案無法續跑。可用的優先序：
#   resume_from.pt（callback 在 strip 之前留的複本）
#   epochN.pt（save_period 產出，不會被 strip，取 N 最大者）
#   last.pt（只有在被強制中斷、來不及 strip 時才可用）
def usable(path):
    try:
        ck = torch.load(path, map_location="cpu", weights_only=False)
    except Exception as e:
        print(f"    {os.path.basename(path)}: 讀取失敗 {type(e).__name__}")
        return None
    ep, opt = ck.get("epoch", -1), ck.get("optimizer")
    del ck
    return ep + 1 if (ep is not None and ep >= 0 and opt is not None) else None

if CKPT_NAME:                      # 使用者指定，跳過自動挑選
    cands = [os.path.join(WDIR, CKPT_NAME)]
    assert os.path.exists(cands[0]), f"找不到指定的 {CKPT_NAME}，該目錄有：{sorted(os.listdir(WDIR))}"
else:
    cands = [os.path.join(WDIR, "resume_from.pt")]
cands += [] if CKPT_NAME else sorted((os.path.join(WDIR, f) for f in os.listdir(WDIR)
                 if re.fullmatch(r"epoch\d+\.pt", f)),
                key=lambda p: int(re.search(r"\d+", os.path.basename(p)).group()), reverse=True)
if not CKPT_NAME:
    cands.append(os.path.join(WDIR, "last.pt"))

CKPT, DONE = None, None
for c in cands:
    if not os.path.exists(c):
        continue
    n = usable(c)
    print(f"    {os.path.basename(c):<18} " + (f"已完成 {n} 輪，可續跑" if n else "已被 strip，不可續跑"))
    if n and CKPT is None:
        CKPT, DONE = c, n

assert CKPT, (
    "沒有任何可續跑的 checkpoint —— 所有權重的 epoch 都是 -1，代表上一場的訓練迴圈"
    "正常結束並執行了 strip_optimizer()。若那是因為早停觸發，表示訓練已收斂，不需要續跑。")
print(f"▷ 2/4 選用 {os.path.basename(CKPT)}（已完成 {DONE} 輪，將續跑到第 {EPOCHS} 輪）")

# ── 3. 還原自訂 loss 的離群度統計量 ───────────────────────────────────
# WIoU v3 的 iou_mean 是滑動平均，不在 checkpoint 裡。少了這步第二場會從 1.0
# 重新暖機，focus 係數與第一場對不上。內建 CIoU 的臂不需要這一步。
if patched:
    ws = os.path.join(WDIR, "wiou_state.txt")
    if os.path.exists(ws):
        v9.set_wiou_state(float(open(ws).read().strip()))
        print(f"▷ 3/4 已還原 WIoU 離群度統計量：{v9.wiou_state():.4f}")
    else:
        print("▷ 3/4 找不到 wiou_state.txt，離群度統計量將從 1.0 重新暖機")
else:
    print("▷ 3/4 本臂使用內建 CIoU，無狀態需要還原")

# ── 3.5 繞過 ultralytics 8.4.121 的續跑 bug ──────────────────
# check_resume() 只重設 args.model / args.resume，**沒有**重設 args.pretrained；
# 而 self.args = get_cfg(ckpt_args) 會把 args.pretrained 從 ckpt 還原成上一場的
# "yolo26n.pt"（字串）。接著 setup_model() 裡那兩段是 if 不是 elif：
#     if str(self.model).endswith(".pt"):                weights = 讀 ckpt          ← 對
#     if isinstance(self.args.pretrained, (str, Path)):  weights = 讀 pretrained    ← 盖掉
# 結果：epoch / optimizer / LR 排程都正確續上了，**但模型權重被 COCO 預訓練權重蓋掉**。
# 征狀：log 印出 "Transferred 360/902"（正確的續跑應該是 902/902），
# 第一個 batch 的 cls_loss 會高達 20以上（正常應該跟上一場收尾相當）。
# 修法：把 ckpt 裡的 pretrained 改成 False，setup_model 就會保留 ckpt 權重。
# （pretrained 不在 check_resume 的可覆寫白名單裡，所以傳 pretrained=False 給
#   .train() 沒用，必須改 ckpt 本體。）
_ck = torch.load(CKPT, map_location="cpu", weights_only=False)
if _ck.get("train_args", {}).get("pretrained") not in (False, None):
    _old = _ck["train_args"]["pretrained"]
    _ck["train_args"]["pretrained"] = False
    CKPT = CKPT[:-3] + "_resumable.pt"
    torch.save(_ck, CKPT)
    print(f"▷ 3.5/4 已把 pretrained（{_old}）改成 False → {os.path.basename(CKPT)}")
else:
    print("▷ 3.5/4 ckpt 的 pretrained 已經是 False，不需要修正")
del _ck

# ── 4. 續跑 ───────────────────────────────────────────────────────────
# resume=True 會沿用 ckpt 內記錄的全部超參數（epochs、batch、lr、close_mosaic…），
# 所以下方不要再傳任何訓練參數。
model = YOLO(CKPT)
print(f"▷ 4/4 開始續跑：第 {DONE + 1} 輪 → 第 {EPOCHS} 輪（還有 {EPOCHS - DONE} 輪）")
results = model.train(resume=True)
# 驗收：上面那行 "Transferred X/902" 必須是 902/902。
# 若印出 360/902，代表 pretrained 修正沒生效，權重被 COCO 預訓練模型蓋掉，
# 這一場算無效 —— 立刻中斷，不要讓它白跑完。

print("▷ RESUME 訓練完畢")

# Step.7 消融分析包
### 與 `train_ablation.ipynb` 同一份，續跑完照樣產出可比對的 zip。

In [ ]:
import csv, json, os, shutil, zipfile

# 只用標準庫，不依賴 pandas —— ultralytics 本身也沒有強制要求它
def write_csv(path, fieldnames, rows):
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)


# 用 best.pt 重跑一次驗證，直接拿到 metrics 與混淆矩陣物件。
# 訓練結束時 Model.train() 已把 self.model 換成 best.pt，所以這裡驗的就是 best。
#
# plots=True 是必要的，不是為了畫圖：ultralytics 把 confusion_matrix.process_batch
# 包在 `if self.args.plots` 裡（detect/val.py:196），plots=False 會讓混淆矩陣
# 維持全零，Scale_Insect 的 FP 數就永遠是 0。
m = model.val(data=DATA_YAML, imgsz=HP["imgsz"], batch=HP["batch"], plots=True)

# 訓練輸出目錄：model.trainer 在 train() 之後仍在，save_dir 一定有效
RUN_DIR = str(model.trainer.save_dir)
OUT = f"/kaggle/working/ablation_{ARM}"
os.makedirs(OUT, exist_ok=True)

# ── 1. 逐輪指標與超參數（直接複製）──────────────────────────────────
for fn in ("results.csv", "args.yaml"):
    src = os.path.join(RUN_DIR, fn)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(OUT, fn))

# ── 2. 每類指標 ────────────────────────────────────────────────────
names = m.names if isinstance(m.names, dict) else {i: n for i, n in enumerate(m.names)}
COLS = ["class_id", "class", "precision", "recall", "f1", "ap50", "ap50_95"]
per_class = []
for i, ci in enumerate(m.box.ap_class_index):
    ci = int(ci)
    per_class.append({
        "class_id": ci,
        "class": names.get(ci, str(ci)),
        "precision": round(float(m.box.p[i]), 5),
        "recall": round(float(m.box.r[i]), 5),
        "f1": round(float(m.box.f1[i]), 5),
        "ap50": round(float(m.box.ap50[i]), 5),
        "ap50_95": round(float(m.box.ap[i]), 5),
    })
per_class.sort(key=lambda r: r["class_id"])
write_csv(os.path.join(OUT, "per_class.csv"), COLS, per_class)

# ── 3. 混淆矩陣原始數字（列=預測，欄=真實，最後一列/欄為背景）──────
cm = m.confusion_matrix.matrix
nc = len(names)
labels = [names.get(i, str(i)) for i in range(nc)] + ["background"]
with open(os.path.join(OUT, "confusion_matrix.csv"), "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow([""] + [f"true_{l}" for l in labels])
    for i, lab in enumerate(labels):
        w.writerow([f"pred_{lab}"] + [int(cm[i][j]) for j in range(len(labels))])

# 每類的 FP / FN（背景那一列/欄）—— v8 報告裡 Scale_Insect 的 615 FP 就是這樣算的
fp_bg = {labels[i]: int(cm[i][nc]) for i in range(nc)}
fn_bg = {labels[i]: int(cm[nc][i]) for i in range(nc)}

# ── 4. F1-信心曲線與最佳截斷點 ──────────────────────────────────────
best_conf = None
try:
    x, y, _xl, _yl = m.curves_results[1]        # F1-Confidence(B)
    x = [float(v) for v in x]
    mean_f1 = ([sum(col) / len(col) for col in zip(*y)] if hasattr(y[0], "__len__")
               else [float(v) for v in y])
    with open(os.path.join(OUT, "f1_conf.csv"), "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(["conf", "mean_f1"])
        w.writerows(zip(x, mean_f1))
    best_conf = round(x[mean_f1.index(max(mean_f1))], 4)
except Exception as e:
    print(f"▷ F1-conf 曲線取用失敗（不影響主判準）：{type(e).__name__}: {e}")

# ── 5. 平台期統計 —— 這是主判準，直接算好省得事後再算 ────────────────
plateau = {}
rcsv = os.path.join(OUT, "results.csv")
if os.path.exists(rcsv):
    with open(rcsv, encoding="utf-8") as f:
        rec = [{k.strip(): v for k, v in row.items()} for row in csv.DictReader(f)]
    win = 50 if EPOCHS >= 120 else 16          # 長跑取最後 50 輪，短跑取最後 16 輪
    tail = rec[-win:]
    for key, col in (("mAP50", "metrics/mAP50(B)"), ("mAP50_95", "metrics/mAP50-95(B)")):
        if rec and col in rec[0]:
            vals = [float(r[col]) for r in tail]
            mean = sum(vals) / len(vals)
            var = sum((v - mean) ** 2 for v in vals) / len(vals)
            plateau[key] = {
                "window": f"last {len(vals)} epochs",
                "mean": round(mean, 5),
                "std": round(var ** 0.5, 5),
                "max_all_epochs": round(max(float(r[col]) for r in rec), 5),
            }
    if rec and "time" in rec[0]:
        plateau["s_per_epoch"] = round(float(rec[-1]["time"]) / len(rec), 1)

summary = {
    "arm": ARM,
    "desc": v9.ARMS[ARM]["desc"],
    "epochs": EPOCHS,
    "arch": v9.ARMS[ARM]["arch"],
    "loss_ratio": v9.ARMS[ARM]["loss_ratio"],
    "hyperparams": {k: (v if isinstance(v, (int, float, bool, type(None))) else str(v))
                    for k, v in sorted(HP.items())},
    "final_best_pt": {"mAP50": round(float(m.box.map50), 5),
                      "mAP50_95": round(float(m.box.map), 5),
                      "precision": round(float(m.box.mp), 5),
                      "recall": round(float(m.box.mr), 5)},
    "plateau": plateau,
    "best_f1_conf": best_conf,
    "fp_from_background": fp_bg,
    "fn_to_background": fn_bg,
    "speed_ms": {k: round(float(v), 3) for k, v in m.speed.items()},
}
with open(os.path.join(OUT, "summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

# ── 6. 打包成 Kaggle output 的 zip ──────────────────────────────────
ZIP = f"/kaggle/working/ablation_{ARM}.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(OUT)):
        z.write(os.path.join(OUT, fn), arcname=f"{ARM}/{fn}")

print("═" * 72)
print(f"▷ 分析包：{ZIP}  ({os.path.getsize(ZIP) / 1024:.1f} KB)")
print(f"   內含 {sorted(os.listdir(OUT))}")
print("═" * 72)
for k, v in plateau.items():
    if isinstance(v, dict):
        print(f"  {k:<10} 平台期({v['window']}) 平均={v['mean']:.5f}  σ={v['std']:.5f}"
              f"   全程最大={v['max_all_epochs']:.5f}")
    else:
        print(f"  {k:<10} {v}")
print(f"  最佳 F1 截斷點 conf = {best_conf}")

print(f"\n{'類別':<20}{'P':>9}{'R':>9}{'F1':>9}{'AP50':>9}{'AP50-95':>10}")
for r in per_class:
    print(f"{r['class']:<20}{r['precision']:>9.4f}{r['recall']:>9.4f}{r['f1']:>9.4f}"
          f"{r['ap50']:>9.4f}{r['ap50_95']:>10.4f}")

print(f"\n  背景誤報 FP：{fp_bg}")
print(f"  漏檢至背景 FN：{fn_bg}")
print(f"\n▷ 下載 ablation_{ARM}.zip 即可，不需要整包 runs_{ARM}.zip")

# Step.6 完整輸出打包

In [ ]:
import os, shutil

runs_dir = "/kaggle/working/runs"
if os.path.exists(runs_dir):
    shutil.make_archive(f"/kaggle/working/runs_{ARM}", "zip", runs_dir)
    size = os.path.getsize(f"/kaggle/working/runs_{ARM}.zip") / (1024 ** 2)
    print(f"▷ runs_{ARM}.zip ({size:.2f} MB)  —— 含權重，供再次續跑或最終交付")
    print(f"▷ ablation_{ARM}.zip —— 判準比對用的小包，優先下載這個")
else:
    print(f"▷ 找不到 {runs_dir}")